In [ ]:
# Zopedia — Wiki Link Graph Exploration
#
# This notebook lets you:
# 1. Load the wiki engine and build the full link graph (all pages)
# 2. Extract the induced subgraph on entity/concept nodes
# 3. Run community detection, god nodes, surprising connections
# 4. Visualize the graph structure
#
# Usage: Run cells in order. Modify parameters in the last sections
# to experiment with different graph algorithms.

## 1. Setup — Initialize Wiki Engine

In [65]:
from __future__ import annotations
import sys
import os

# Add backend to path
sys.path.insert(0, os.path.abspath("../backend"))
sys.path.insert(0, os.path.abspath("/Users/zohairshafi/Local Workspace/zopedia/backend/core/"))

from pathlib import Path
from core.wiki.engine import LLMWikiEngine, WikiConfig

# Point to the wiki vault
VAULT_ROOT = Path(os.path.expanduser("~/unsloth_wiki"))
WIKI_DIR = VAULT_ROOT / "wiki"

print(f"Vault root: {VAULT_ROOT}")
print(f"Wiki dir:  {WIKI_DIR}")
print(f"Exists:    {WIKI_DIR.exists()}")

Vault root: /Users/zohairshafi/unsloth_wiki
Wiki dir:  /Users/zohairshafi/unsloth_wiki/wiki
Exists:    True


## 2. Build the FULL Link Graph

Build the graph from ALL pages (entities, concepts, analysis, sources).
This preserves multi-hop connections where analysis/source pages act as
bridges between entity/concept nodes.

In [66]:
# Create engine with default config
cfg = WikiConfig(vault_root=VAULT_ROOT)
engine = LLMWikiEngine(llm_fn=lambda prompt: "", cfg=cfg)

# ALL pages — not just entities/concepts
all_pages = engine._all_wiki_pages()
print(f"Total pages: {len(all_pages)}")

# Categorize
by_type = {"entities": [], "concepts": [], "analysis": [], "sources": [], "other": []}
for p in all_pages:
    if p.startswith("entities/"):
        by_type["entities"].append(p)
    elif p.startswith("concepts/"):
        by_type["concepts"].append(p)
    elif p.startswith("analysis/"):
        by_type["analysis"].append(p)
    elif p.startswith("sources/"):
        by_type["sources"].append(p)
    else:
        by_type["other"].append(p)

for k, v in by_type.items():
    print(f"  {k}: {len(v)}")

Total pages: 1506
  entities: 538
  concepts: 481
  analysis: 235
  sources: 249
  other: 3


## 3. Build Full Link Graph

`_build_link_graph` parses all [[wikilinks]] and returns:
- `inbound`: {page: [pages that link TO it]}
- `outbound`: {page: [pages it links TO]}
- `broken`: [(source, target)] for broken links

In [67]:
full_graph = engine._build_link_graph(all_pages)

print("Graph keys:", list(full_graph.keys()))
print(f"Inbound entries:  {len(full_graph.get('inbound', {}))}")
print(f"Outbound entries: {len(full_graph.get('outbound', {}))}")
print(f"Broken links:     {len(full_graph.get('broken', []))}")

Graph keys: ['inbound', 'outbound', 'broken']
Inbound entries:  1506
Outbound entries: 1506
Broken links:     162


In [68]:
import networkx as nx 
from networkx.algorithms import bipartite

all_edges = full_graph.get("outbound", {}) | full_graph.get("inbound", {})

entity_nodes = [x for x in list(all_edges.keys()) if x.startswith("entities/")]
concept_nodes = [x for x in list(all_edges.keys()) if x.startswith("concepts/")]
analysis_nodes = [x for x in list(all_edges.keys()) if x.startswith("analysis/")]
source_nodes = [x for x in list(all_edges.keys()) if x.startswith("source/")]

print (len(entity_nodes), len(concept_nodes), len(analysis_nodes), len(source_nodes))


538 481 235 0


In [69]:
entity_graph = nx.Graph()
concept_graph = nx.Graph()

# Add nodes with the node attribute "bipartite"
entity_graph.add_nodes_from(entity_nodes, bipartite=0)
entity_graph.add_nodes_from(analysis_nodes + source_nodes, bipartite=1)

concept_graph.add_nodes_from(concept_nodes, bipartite=0)
concept_graph.add_nodes_from(analysis_nodes + source_nodes, bipartite=1)

concept_bipartite_edges = []
entity_bipartite_edges = []

for node, neighbors in all_edges.items():
    if node in entity_nodes:
        for neighbor in neighbors:
            if neighbor in analysis_nodes + source_nodes:
                entity_bipartite_edges.append((node, neighbor))
    elif node in concept_nodes:
        for neighbor in neighbors:
            if neighbor in analysis_nodes + source_nodes:
                concept_bipartite_edges.append((node, neighbor))
    # Analysis / source nodes
    else: 
        for neighbor in neighbors:
            if neighbor in entity_nodes:
                entity_bipartite_edges.append((neighbor, node))
            elif neighbor in concept_nodes:
                concept_bipartite_edges.append((neighbor, node))


# Add edges only between nodes of opposite node sets
entity_graph.add_edges_from(entity_bipartite_edges)
concept_graph.add_edges_from(concept_bipartite_edges)

In [70]:
entity_projection = bipartite.projected_graph(entity_graph, entity_nodes)
concept_projection = bipartite.projected_graph(concept_graph, concept_nodes)

In [ ]:
concept_communities = nx.community.greedy_modularity_communities(concept_projection, cutoff = 20)
entity_communities = nx.community.greedy_modularity_communities(entity_projection, cutoff = 20)

# Keep only communities above a certain size and merge the rest into an "other" bucket
community_cutoff = 10
concept_communities = [c for c in concept_communities if len(c) >= community_cutoff]
entity_communities = [c for c in entity_communities if len(c) >= community_cutoff]


In [102]:
concept_community_names = []
for community in concept_communities:
    name = ""
    for node in community:
        name += f"  {node}\n"
    concept_community_names.append(name)

entity_community_names = []
for community in entity_communities:
    name = ""
    for node in community:
        name += f"  {node}\n"
    entity_community_names.append(name)

# Get an LLM to name the communities
